# SQF Optimisation — Interactive Notebook

This notebook provides a guided workflow for score-based chromatographic method optimisation from scouting data. Using a configuration file and a measurements file, it fits predictive models, evaluates the explored method space, computes separation scores, visualises the resulting score surfaces, and identifies promising operating conditions.

For most users, the notebook is intended to be run from top to bottom with minimal edits. In standard use, the main change required is to provide your own input files in Section 1 by updating the file paths for the configuration file and the measurements file.

If you are using the Binder version of this notebook, your own files can be uploaded directly through the JupyterLab interface. To do so, open the file browser on the left side of the window if it is collapsed, navigate to the folder where you want to place your files, and use the **Upload Files** button or drag and drop the files into the file browser. For simplicity, it is recommended to upload them either into the `data/` folder or into the same directory as this notebook, then copy the corresponding paths into the code cell in Section 1.

If you are not familiar with Jupyter notebooks, the recommended approach is simply to run the cells in order from the top of the document to the bottom. In JupyterLab, this can be done cell by cell, or by using **Run > Run All Cells** after the input file paths have been set correctly. Another recommended action is to turn on **Render side-by-side** from the **View** menu, to see more easily the results of code blocks.

The optional final section is not required for the standard workflow. It is provided for manual exploration of a specific condition and as a flexible workspace for more experienced users who may wish to add their own code below.

**Sections**
- 1. Initialisation and Input Data
- 2. Model fitting
- 3. Standard Workflow
  - 3.1. Prediction and Score Computation
  - 3.2. Visualisation
  - 3.3. Optimisation and Simulation
- 4. Optional: Simulation at an arbitrary point

## Initialisation and Input Data

This section loads the two input files required for the analysis:

- a **configuration file**, which defines the instrument and method parameters used both for the predictive domain and score calculations,
- a **measurements file**, which contains the scouting experimental data used to fit the predictive models.

For most users, this is the only part of the notebook that needs to be edited before running the standard workflow. In the code cell below, replace the example file paths with the paths to your own configuration and measurements files if needed.

Example input files are provided in the `data/` folder. If you are using this notebook for the first time, it is recommended to start with those examples and confirm that the notebook runs correctly from top to bottom before loading your own data.

If you are using the Binder version of this notebook, you can upload your own files directly in the JupyterLab file browser. After upload, make sure to note their location and update the file paths in the code cell below accordingly. To keep things simple, it is recommended to upload both files into the `data/` folder or into the notebook’s root directory.

After running the next cell, the configuration and measurements will be loaded into memory and used by the following sections.

In [ ]:
from sqf_optimisation.io import load_config, load_measurements

# Edit the paths to the config and measurements files as needed

config = load_config("data/instrument-config_5_amlodipine.json")
measurements = load_measurements("data/measurements_5_amlodipine.csv")


## Model fitting

This section fits the predictive models used throughout the rest of the notebook. Using the scouting measurements loaded above, the program estimates per-analyte model parameters describing retention behaviour and peak width as functions of the method conditions.

These fitted parameters form the basis for the later prediction and optimisation steps. In the standard workflow, no user action is required here beyond running the code cell below.

As with any model-based approach, the quality of the predictions depends on the quality and consistency of the input data. The results of the following sections should therefore be interpreted as model-based estimates to guide method development, not as a substitute for experimental confirmation.

In [ ]:
from sqf_optimisation.core import fit_models

model_params = fit_models(config, measurements)


## Standard Workflow

This section contains the main analysis workflow of the notebook. Using the fitted model parameters, the program predicts chromatographic behaviour across the method space, computes the associated scores, visualises the resulting surfaces, and identifies promising operating conditions.

The following subsections correspond to the main stages of this workflow.

### Prediction and Score Computation

This step runs the core analysis of the notebook. Using the fitted model parameters from Section 2, the program evaluates the full method space defined in the configuration file, predicts chromatographic behaviour at each point, and computes the corresponding score surfaces.

The output of this step consists of two objects: a prediction grid containing the simulated chromatographic quantities, and a score grid containing the derived subscores and the global SQF score. These results are used directly in the following subsections for visualisation and optimisation.

The fineness of these evaluation grids is controlled by the `n_T` (for the temperature axis) and `n_tG`(for the gradient time axis) parameters in the configuration file. Larger values produce a denser grid and smoother score surfaces, but also increase computation time.

In the standard workflow, no user action is required here beyond running the code cell below.


In [ ]:
from sqf_optimisation.core import run_grid_analysis

pred_grid, scores_grid = run_grid_analysis(config, model_params)


### Visualisation

This subsection displays the score surfaces computed in the previous step. These figures provide a graphical overview of how the different separation criteria, as well as the global SQF score, vary across the method space.

In general, higher values indicate more promising predicted conditions. The individual subscore surfaces can be used to understand which criteria drive the optimisation, while the SQF surface provides a global summary of the predicted separation quality over the full domain.

In [ ]:
from sqf_optimisation.visualisation import plot_subscore_surfaces, plot_sqf_surface

plot_subscore_surfaces(scores_grid).show()
plot_sqf_surface(scores_grid).show()

### Optimisation and Simulation

This subsection uses the computed SQF surface to identify the best predicted operating point within the explored method space. The corresponding score values are then displayed, together with a synthetic chromatogram generated from the model predictions at that condition.

This provides a practical summary of the optimisation result: both a numerical qualification and a visual representation of the expected separation at that point. As in the previous sections, these results are model-based predictions and should be interpreted as guidance for experimental method development rather than as final validated conditions.

In [ ]:
from sqf_optimisation.core import find_optimum

T_max, tG_max, _, i_T, i_tG = find_optimum(scores_grid)

print(f"Optimum found at T={T_max:.2f} K, tG={tG_max:.2f} min:\n")
print(scores_grid.isel(T=i_T, tG=i_tG).to_pandas().to_string())


In [ ]:
from sqf_optimisation.visualisation import plot_synthetic_chromatogram

plot_synthetic_chromatogram(pred_grid, i_T, i_tG, measurements).show()


## Optional: Simulation at an arbitrary point

This optional section allows the user to evaluate a chosen condition instead of the predicted optimum. It is not required for the standard workflow, but can be useful for manually exploring specific temperature / gradient-time combinations or for comparing a known experimental setting with the model predictions.

To use this section, edit the values of `T_test` and `tG_test` in the code cell below. The notebook will then predict the chromatographic behaviour at that point, compute the corresponding scores, and display a synthetic chromatogram for visual inspection.

This section can also serve as a flexible workspace for more experienced users, who may freely add code cells below to inspect the results further, test alternative calculations, or adapt the workflow to their own needs.

In [ ]:
import numpy as np
from sqf_optimisation.core import predict_grid_from_params, compute_scores

# ---- Choose a test point (edit as desired) ----
T_test = float(config.T1)   # or any temperature in your domain
tG_test = float(config.tG1) # or any gradient time in your domain

# ---- Predict ----
pred_sp= predict_grid_from_params(
    model_params,
    T_vals=np.array([T_test], dtype=float),
    tG_vals=np.array([tG_test], dtype=float),
    t0_total=float(config.t0_total_min),
)

# ---- Compute scores ----
scores_sp = compute_scores(
    pred_sp,
    column_dead_time=float(config.t0_col_min),
    width_penalty_coeff=10.0,
)

# ---- Display results ----
display(scores_sp)

In [ ]:
from sqf_optimisation.visualisation import plot_synthetic_chromatogram

plot_synthetic_chromatogram(pred_sp, 0, 0, measurements).show()